<div dir="rtl" style="text-align: right;">
<h1>למידת מכונה - פרויקט</h1>
</div>

<div dir="rtl">
</div>
<div dir="rtl">
<h2>
חלק 1 - הקדמה
</h2>

פרטי הסטודנטים:
נויה א 6233, עמית א 9054 ואמיר מ 8889.

שימוש בכלי AI (פרומפטים):


הסבר על בעיית הלמידה וה-dataset:
בחרנו ב-dataset מתוך Kaggle בשם "Emotions Detection". מסד נתונים זה מכיל משפטים קצרים ומתייג כל אחד מהם לרגש מסוים. המטרה היא לאמן מודל למידת מכונה שיוכל לסווג טקסטים גולמיים לרגש המתאים להם. מדובר בבעיית סיווג רב-מחלקתית (Multi-class Classification) בתחום ניתוח טקסט (NLP).

</div>

<div dir="rtl" style="text-align: right;">
<h3>הבעיה וה-Dataset</h3>
<p>המטלה עוסקת בבעיית סיווג רב-מחלקתי  בתחום ניתוח הטקסט (NLP): בהינתן משפט יוחזר הרגש שנובע ממנו.

 ה-Dataset המקורי (<code>emotions-detection-text-dataset</code>) מכיל משפטים באנגלית שכל אחד מתויג ברגש אחד מתוך שש קטגוריות:
 anger, fear, joy, love, sadness, surprise.

  בפורמט <code>text;emotion</code>. מכיוון שהקובץ המקורי מגיע כקובץ יחיד ללא חלוקת train/test מובנית, ביצענו חלוקה חד-פעמית ל-train ו-test מיד בשלב הטעינה.  </p>
</div>

<div dir="rtl" style="text-align: right;">
<h2>טעינת ה-Dataset</h2>
</div>

In [27]:
%pip install -q kagglehub

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 100)

In [28]:

dataset_path = kagglehub.dataset_download("abhrajaiswal/emotions-detection-text-dataset")

df = pd.read_csv(f"{dataset_path}/emotions.txt", sep=";", names=["text", "emotion"])
print(f"Total rows in original dataset: {len(df)}")
df.head()

Using Colab cache for faster access to the 'emotions-detection-text-dataset' dataset.
Total rows in original dataset: 16000


,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned hopeful just from being around someone who cares ...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplace i will know that it is still on the property,love
4,i am feeling grouchy,anger


In [29]:
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["emotion"], random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train set: {len(train_df)} rows | Test set: {len(test_df)} rows")

Train set: 12800 rows | Test set: 3200 rows


In [30]:
print("First 5 rows of the Train set:")
display(train_df.head())

First 5 rows of the Train set:


,text,emotion
0,i was feeling drained before i even sat in the chair,sadness
1,i alsways feel so carefree,joy
2,i dont know about you guys but i certainly feel fabulous about myself,joy
3,i also learned that when i feel passionate about what i m writing i can actually be quite good a...,joy
4,i feel like im loving them even more now that im working again i appreciate every snuggle and fe...,love


In [31]:
print("First 5 rows of the Test set:")
display(test_df.head())

First 5 rows of the Test set:


,text,emotion
0,i feel wronged by certain people and my instinct was to get angry at them and stop speaking to t...,anger
1,i feel so calm with the routine rinse wash with detergent rinse take outside to line dry,joy
2,i feel like im not welcomed here i just dont like blend in or something,joy
3,i feel totally listless exams have come and gone and now i have a whole five or so months in fro...,sadness
4,i am feeling confident that i will be able to get to the back door before dinner time,joy


<div dir="rtl" style="text-align: right;">
<h2>מדד האיכות</h2>
<p>מדובר בבעיית סיווג רב-מחלקתי  עם 6 מחלקות (anger, fear, joy, love, sadness, surprise) ללא מחלקה מרכזית אחת. לכן, לפי הנחיות המטלה, מדד האיכות בו נשתמש הוא macro-average F1  ממוצע (לא משוקלל) של ציון ה-F1 שמחושב בנפרד לכל מחלקה. בחירה זו מתאימה כיוון שהיא נותנת משקל שווה לכל רגש, גם לרגשות נדירים יחסית ב-dataset (כמו surprise ו-love), ולא רק לרגשות הנפוצים (כמו joy ו-sadness).</p>
</div>

In [32]:
from sklearn.metrics import f1_score, classification_report

def evaluate(y_true, y_pred, title=""):
    score = f1_score(y_true, y_pred, average="macro")
    print(f"{title} macro-F1: {score:.4f}")
    return score

<div dir="rtl" style="text-align: right;">
<h2>חלק 2 - Feature Engineering</h2>
<p>הטקסט לא קביל ישירות לאלגוריתם למידה, לכן נהפוך כל משפט לוקטור מספרי בשיטת Bag-of-Words (וקטוריזציה על בסיס ספירת מילים) שנלמדה בכיתה: כל מאפיין  מייצג מילה מהאוצר מילים , והערך שלו הוא מספר הפעמים שהמילה מופיעה במשפט. אנו מסננים מילות עצירה נפוצות  שאינן נושאות מידע על הרגש (כגון "the", "is", "and"). זהו הבסיס הטבעי לאלגוריתם Naive Bayes המשתמש בשכיחויות מילים לכל מחלקה.</p>
<p>ה-vectorizer מותאם (fit) רק על ה-trainset, ולאחר מכן משמש להמרת (transform) גם את ה-trainset וגם את ה-testset  כך נמנעים מדליפת מידע מה-testset.</p>
</div>

In [33]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])
y_train = train_df["emotion"].values
y_test = test_df["emotion"].values

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} words")
print(f"Train matrix shape: {X_train.shape} | Test matrix shape: {X_test.shape}")

Vocabulary size: 13225 words
Train matrix shape: (12800, 13225) | Test matrix shape: (3200, 13225)


<div dir="rtl" style="text-align: right;">
<h3>הדגמת ה-Feature Engineering על 2-3 דוגמאות</h3>
</div>

In [34]:
def show_bow_example(text_series, X, idx):
    row = X[idx]
    words = vectorizer.get_feature_names_out()[row.indices]
    counts = row.data
    print(f"Original sentence: {text_series.iloc[idx]!r}")
    print("Bag-of-Words representation (word: count):", dict(zip(words, counts)))
    print()

print("--- Train examples ---")
for i in [0, 1, 2]:
    show_bow_example(train_df["text"], X_train, i)

print("--- Test examples ---")
for i in [0, 1]:
    show_bow_example(test_df["text"], X_test, i)

--- Train examples ---
Original sentence: 'i was feeling drained before i even sat in the chair'
Bag-of-Words representation (word: count): {'feeling': np.int64(1), 'drained': np.int64(1), 'sat': np.int64(1), 'chair': np.int64(1)}

Original sentence: 'i alsways feel so carefree'
Bag-of-Words representation (word: count): {'alsways': np.int64(1), 'feel': np.int64(1), 'carefree': np.int64(1)}

Original sentence: 'i dont know about you guys but i certainly feel fabulous about myself'
Bag-of-Words representation (word: count): {'feel': np.int64(1), 'dont': np.int64(1), 'know': np.int64(1), 'guys': np.int64(1), 'certainly': np.int64(1), 'fabulous': np.int64(1)}

--- Test examples ---
Original sentence: 'i feel wronged by certain people and my instinct was to get angry at them and stop speaking to them but two wrongs dont make a right i think'
Bag-of-Words representation (word: count): {'angry': np.int64(1), 'certain': np.int64(1), 'dont': np.int64(1), 'feel': np.int64(1), 'instinct': np.int

<div dir="rtl" style="text-align: right;">
<h2>חלק 3 - מימוש אלגוריתם למידה: Multinomial Naive Bayes</h2>
<p>בחרנו ב Naive Bayes כי הוא אלגוריתם גנרטיבי ומתאים באופן טבעי לנתוני טקסט המיוצגים כספירות מילים (Bag-of-Words): הוא מניח (בנאיביות) שכל מילה במשפט תורמת עדות בלתי-תלויה לרגש שלו, ומחשב לפי חוק בייס את ההסתברות של כל מחלקה (רגש) בהינתן המילים שבמשפט.</p>

<p><b>Hyperparameter:</b> <code>alpha</code>
 מקדם ה-Laplace smoothing (ברירת מחדל 1.0). ככל ש-alpha גדול יותר, כך ה הסתברויות "מוחלקות" יותר לכיוון אחיד בין המילים.</p>
</div>

In [35]:
import numpy as np

class MultinomialNaiveBayes:
    """מימוש עצמי של Multinomial Naive Bayes, ל-Bag-of-Words features."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_docs, n_features = X.shape
        self.class_log_prior_ = np.zeros(len(self.classes_))
        self.feature_log_prob_ = np.zeros((len(self.classes_), n_features))
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_log_prior_[i] = np.log(X_c.shape[0] / n_docs)
            word_counts = np.asarray(X_c.sum(axis=0)).flatten() + self.alpha
            self.feature_log_prob_[i] = np.log(word_counts / word_counts.sum())
        return self

    def predict_log_proba(self, X):
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict(self, X):
        log_proba = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_proba, axis=1)]

<div dir="rtl" style="text-align: right;">
<h3>בדיקת תקינות (sanity check)</h3>
<p>נוודא שהמימוש העצמי שלנו מניב תוצאות זהות (או קרובות מאוד) למימוש הבנוי-מראש של <code>sklearn.naive_bayes.MultinomialNB</code>, כאשר משתמשים באותו alpha.</p>
</div>

In [36]:
from sklearn.naive_bayes import MultinomialNB

_our_model = MultinomialNaiveBayes(alpha=1.0).fit(X_train, y_train)
_sklearn_model = MultinomialNB(alpha=1.0).fit(X_train, y_train)

_our_pred = _our_model.predict(X_test)
_sklearn_pred = _sklearn_model.predict(X_test)

agreement = (_our_pred == _sklearn_pred).mean()
print(f"Agreement between our implementation and sklearn: {agreement:.2%}")
assert agreement > 0.99, "Our implementation deviates significantly from sklearn - check the code"

Agreement between our implementation and sklearn: 100.00%


<div dir="rtl" style="text-align: right;">
<h2>חלק 4 - אימון: הפעלת ה-flow לפי פרמטרים שונים</h2>
<p>פונקציית ה-<code>fit</code> שממומשת למעלה תומכת בכל ערך של alpha. להלן דוגמה להרצת האימון עם כמה ערכי alpha שונים, ולבחירת המודל הסופי שיאומן על כל ה-trainset. (השוואה שיטתית ומלאה של hyperparameters, בעזרת grid-search ו-k-fold cross validation, מתבצעת בסעיף ההרחבה 6.א)</p>
</div>

In [37]:
for alpha_try in [0.1, 1.0, 5.0]:
    model_try = MultinomialNaiveBayes(alpha=alpha_try).fit(X_train, y_train)
    train_pred = model_try.predict(X_train)
    evaluate(y_train, train_pred, title=f"alpha={alpha_try} | train")

# בחרנו alpha=1.0 (ברירת המחדל הקלאסית) ומאמנים את המודל הסופי על כל ה-trainset
final_alpha = 1.0
model = MultinomialNaiveBayes(alpha=final_alpha).fit(X_train, y_train)
print(f"\nFinal model trained with alpha={final_alpha} on {X_train.shape[0]} training examples")

alpha=0.1 | train macro-F1: 0.9616
alpha=1.0 | train macro-F1: 0.8373
alpha=5.0 | train macro-F1: 0.4666

Final model trained with alpha=1.0 on 12800 training examples


<div dir="rtl" style="text-align: right;">
<h2>חלק 5 - חיזוי ושערוך איכות המודל על ה-test set</h2>
</div>

In [38]:
test_predictions = model.predict(X_test)

print("First 5 predictions on the test set:")
for i in range(5):
    print(f"  Sentence: {test_df['text'].iloc[i]!r}")
    print(f"  True emotion: {y_test[i]}  |  Predicted emotion: {test_predictions[i]}")
    print()

First 5 predictions on the test set:
  Sentence: 'i feel wronged by certain people and my instinct was to get angry at them and stop speaking to them but two wrongs dont make a right i think'
  True emotion: anger  |  Predicted emotion: anger

  Sentence: 'i feel so calm with the routine rinse wash with detergent rinse take outside to line dry'
  True emotion: joy  |  Predicted emotion: joy

  Sentence: 'i feel like im not welcomed here i just dont like blend in or something'
  True emotion: joy  |  Predicted emotion: joy

  Sentence: 'i feel totally listless exams have come and gone and now i have a whole five or so months in front of me with no uni and free time'
  True emotion: sadness  |  Predicted emotion: sadness

  Sentence: 'i am feeling confident that i will be able to get to the back door before dinner time'
  True emotion: joy  |  Predicted emotion: joy



In [39]:
test_score = evaluate(y_test, test_predictions, title="Test set")
print()
print(classification_report(y_test, test_predictions))

Test set macro-F1: 0.6331

              precision    recall  f1-score   support

       anger       0.87      0.66      0.75       432
        fear       0.86      0.64      0.74       387
         joy       0.75      0.93      0.83      1072
        love       0.89      0.33      0.48       261
     sadness       0.75      0.93      0.83       933
    surprise       0.73      0.10      0.17       115

    accuracy                           0.78      3200
   macro avg       0.81      0.60      0.63      3200
weighted avg       0.79      0.78      0.76      3200

